# Cumulocity → SAP Business Data Cloud: remove the data product

**Irreversible.** Removes what `cumulocity_bdc_data_product.ipynb` created for the tenant set below. Everything from `cumulocity_build_up.ipynb` stays: the schema-change task keeps rebuilding `CURATED`. Every statement uses `IF EXISTS` or is skipped when its object is gone, so the notebook can be re-run.

| Step | Removes |
|---|---|
| 1. Republish tasks | Tasks `OPS.BDC_CHECK`, `OPS.BDC_PUBLISH`, procedures `OPS.BDC_*`, table `OPS.BDC_STATE` in `CUMULOCITY_<T>` |
| 2. Connector, share, database | Share `share_name` from the BDC connector, the share, shared database `CUMULOCITY_BDC_<T>` |
| 3. Check | Confirms nothing is left |

End the agreement in Datasphere first (*Data Product Access → Agreements →* the agreement *→ End Agreement*). Removing the share from the connector withdraws the data product's data from BDC; installations in other systems stop working. The catalog entry in BDC may remain (SAP documents no unpublish call).

## Parameters

Set `catalog_name` and `share_name` as in `cumulocity_bdc_data_product.ipynb`.

In [ ]:
import re

catalog_name = ""   # Cumulocity tenant ID
share_name   = "CUMULOCITY_ADHOC_DEMO"                            # as in cumulocity_bdc_data_product.ipynb
connector    = "CONNECTORS.ZEROCOPY.DEFAULT_SAP_BDC_CONNECTOR"

assert re.fullmatch(r"[A-Za-z0-9]+", catalog_name), "catalog_name must be alphanumeric (used in object names)"
tenant = catalog_name.upper()
app_db = f"CUMULOCITY_{tenant}"
bdc_db = f"CUMULOCITY_BDC_{tenant}"

print("Will remove:")
for value in [f"{app_db}.OPS.BDC_*", share_name, bdc_db]:
    print("  ", value)

In [ ]:
%%sql -r set_role
USE ROLE ACCOUNTADMIN

## 1. Republish tasks

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Tasks can only leave the graph while its root is suspended. The root (from the build-up notebook)
# keeps rebuilding CURATED, so it is resumed at the end.
ops = f"{app_db}.OPS"
for stmt in [f"ALTER TASK IF EXISTS {ops}.REBUILD_ON_SCHEMA_CHANGE SUSPEND",
             f"DROP TASK IF EXISTS {ops}.BDC_PUBLISH",
             f"DROP TASK IF EXISTS {ops}.BDC_CHECK",
             f"DROP PROCEDURE IF EXISTS {ops}.BDC_PUBLISH()",
             f"DROP PROCEDURE IF EXISTS {ops}.BDC_CSN()",
             f"DROP PROCEDURE IF EXISTS {ops}.BDC_CHECK()",
             f"DROP PROCEDURE IF EXISTS {ops}.BDC_CREATE_TABLES()",
             f"DROP TABLE IF EXISTS {ops}.BDC_STATE",
             f"SELECT SYSTEM$TASK_DEPENDENTS_ENABLE('{ops}.REBUILD_ON_SCHEMA_CHANGE')"]:
    try:
        session.sql(stmt).collect()
        print("done:   ", stmt)
    except Exception as e:
        if "does not exist" not in str(e):
            raise
        print("skipped:", stmt, "(database or task not found)")

## 2. Connector, share, database

In [ ]:
# The connector and the share are account-level, so they are removed explicitly before the databases.
# REMOVE SHARE fails when the share isn't associated (or share-back is off); skip in that case.
for stmt in [f"ALTER ZEROCOPY CONNECTOR {connector} REMOVE SHARE {share_name}",
             f"DROP SHARE IF EXISTS {share_name}",
             f"DROP DATABASE IF EXISTS {bdc_db}"]:
    try:
        session.sql(stmt).collect()
        print("done:   ", stmt)
    except Exception as e:
        if "does not exist" not in str(e) and "not associated" not in str(e).lower():
            raise
        print("skipped:", stmt, f"({str(e).splitlines()[0]})")

## 3. Check

Each cell should return no rows.

In [ ]:
%%sql -r remaining_share
SHOW SHARES LIKE '{{share_name}}'

In [ ]:
%%sql -r remaining_bdc_db
SHOW DATABASES LIKE '{{bdc_db}}'